### <span style="color:orange;">Scope</span>

**Top-in-class** drugs are developed when other medications are already available for a specific condition / patient stratification, but they show significant limitations. 

Examples can include a limited efficacy, presence of side effects or an inconvenient dosing schedule/modality.

This investigation aims at finding areas of unmet clinical need, where despite available treatments, development of new drugs might still be beneficial to address specific issues.

Due to the nature of the investigation, only reviews with commentary will be considered. Across multiple medical conditions, those with lower global ratings and an high number of reviews will be prioritised for further analysis. 

The final investigation on the selected condition showing unmet need should include:
* information on competitor drugs where available in the dataset (e.g. list of treatments used by reviewers)
* possible correlations between a side effect and a specific treatment
* interrogation on benefits reported from treatment (information on needs currently met)
* summary of (potentially recurrent) side effects where applicable

Please note that this analysis is performed on a small set of drug reviews from 2018, so there is no claim for findings to be accurate or exhaustive at any extent.

### <span style="color:orange;">Packages and Data Import</span>

In this section, relevant libraries will be loaded, and the dataset will be imported.

Data from the dataset [Drug Reviews](https://doi.org/10.24432/C55G6J) will directly be imported via the `ucimlrepo` package, as per *Import in Python* snippet on creators' website.

#### <span style="color:#F97316;">Packages installation</span>

In [ ]:
#!pip install bertopic

In [ ]:
#!pip install -U plotly==5.19.0

In [ ]:
#!pip install transformers==4.56.0

In [ ]:
#!pip install wordcloud

In [ ]:
#!pip install nltk

#### <span style="color:#F97316;">Libraries loading</span>

In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"  # avoids a network hang on HF's newer download backend
os.environ["USE_TF"] = "0"              # skip unneeded TensorFlow detection (this project is PyTorch-only)

In [ ]:
import numpy as np      # numerical arrays and math operations
import pandas as pd     # dataframes operations

import matplotlib.pyplot as plt     # visualisations
import seaborn as sns       # higher-level statistical plotting

import json     # reading/writing JSON data
import random   # generating random numbers / sampling

from ucimlrepo import fetch_ucirepo     # Required for loading the specific dataset

from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram

import re       # pattern matching and cleaning text
from collections import Counter # counts item frequencies (e.g. word counts)

import nltk     # for nltk.download() calls below

import torch    # PyTorch (deep learning framework)
from transformers import AutoModelForCausalLM   # loads pretrained generative language model (Hugging Face)
from transformers import AutoTokenizer      # matching tokenizer for that model
from transformers import pipeline       # Hugging Face's high-level shortcut that wraps model and tokenizer together for common tasks
from sentence_transformers import SentenceTransformer   # generates dense sentence/document embeddings (vector representations)
from sklearn.cluster import AgglomerativeClustering     # Deterministic clustering algorithm

In [ ]:
# Download sentences/word tokenizer models
nltk.download('stopwords', quiet=True)  # static word list of NLTK's stopwords
nltk.download('punkt', quiet=True)  # statistical algorithm based on punctuation/capitalization patterns
nltk.download('punkt_tab', quiet=True)  # newer version of 'punkt'
nltk.download('wordnet', quiet=True)    # for lemmatizer use

In [ ]:
# Set random seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.random.manual_seed(SEED)

In [ ]:
# Ahead of import of functions in the src folder, this enables the system to pick up edits to functions.py without needing to restart the kernel
%load_ext autoreload
%autoreload 2

In [ ]:
# Get access to functions in src folder
import sys
sys.path.append('src')

from functions import *

#### <span style="color:#F97316;">Data import</span>

Please note that the dataset will not be directly visualised in the following sections to adhere to the no-distribution policy from the creators. Only snippets showing aggregated data will be kept.

In [ ]:
# fetch dataset 
drug_reviews_druglib_com = fetch_ucirepo(id=461) 

In [ ]:
# Save dataset in pd dataframe 
X = drug_reviews_druglib_com.data.features 

In [ ]:
# Inspect dataset structure
X.info()

### <span style="color:orange;">Dataset Preparation</span> 

In this section, a copy of the original dataset will be pre-processed in preparation for the analysis.




#### <span style="color:#F97316;">Dataset copy and nulls removal</span>

As I am interested in reviews to identify unmet need per condition, I will drop rows with nulls. 

From the previous `describe()`, it was shown that null entries only affected Condition or Review features, so I can proceed removing all nulls.

In [ ]:
X_reviews = X.dropna()
X_reviews.info()

Now all 4013 available reviews should have commentary in every relevant field.

#### <span style="color:#F97316;">Conditions clustering</span>

I want now to have a feel for how many conditions and drugs are included in the dataset.

In [ ]:
X_reviews.nunique()

As it appears that more conditions (1754) than drugs (534) are listed, it might be that the same condition is reported with slightly different definitions across different reviews. 

I will inspect this theory investigating the `condition` feature.

In [ ]:
cond_counts = X_reviews["condition"].value_counts().sort_values(ascending=False)
cond_counts.head(20)

This seems to confirm the theory (see "*migraine*" vs "*migraines*", or "*high blood pressure*" vs "*hypertension*").

I will procees and create embeddings for each listed condition, then cluster them based on their similarity.

In [ ]:
# Save the unique conditions in an array
conditions = X_reviews["condition"].unique().tolist()
conditions[1:10]

Amongst `sentence-transformers` general purpose models, I am selecting `all-mpnet-base-v2`. 

Compared to the standard lightweight default `all-MiniLM-L6-v2`, it has more layers and can capture more nuances in meaning. This comes at a cost in terms of speed and embeddings size, but given the relatively small dataset it is a reasonable trade-off.

In [ ]:
# Download the model
embedder = SentenceTransformer("all-mpnet-base-v2")

I will now generate the embeddings for the conditions, and save them locally.

In [ ]:
# Generate conditions embeddings
embeddings_cond = embedder.encode(
    conditions,
    show_progress_bar=True,
    batch_size=32
)

In [ ]:
# Cache to disk
np.save("data/embeddings_cond.npy", embeddings_cond)

In [ ]:
# Reload embeddings
embeddings_cond = np.load("data/embeddings_cond.npy")

I will now cluster embeddings to remove redundancies in conditions classifications.

As I don't need keywords to describe clusters' themes, I will not use `BERTopic` at this stage. As I will only work on a subset of conditions, and it's reasonable to verify the quality of clusters for those, I can use simpler clustering methods such as `KMeans`, `HDBSCAN` or `AgglomerativeClustering`.

`AgglomerativeClustering` is a hierarchical, bottom-up clustering method that could fit well where looking for umbrella categories in medical conditions. Moreover, I won't need to set a fixed number of clusters (I can instead cut the tree by distance via `distance_threshold`). It is a deterministic method and supports cosine distance directly (better than Euclidean distance for text embeddings). 

To decide the `distance_threshold`, I will plot a dendrogram of the conditions' embeddings.

In [ ]:
distances = pdist(embeddings_cond, metric='cosine')
Z = linkage(distances, method='average')

plt.figure(figsize=(12, 6))
dendrogram(Z)
plt.ylabel('Distance')
plt.yticks(np.arange(0.1, 1.1, 0.1))
plt.grid(axis='y')
plt.gca().set_xticklabels([])
plt.show()

A distance between 0.3 and 0.4 should be able to cluster nearly-identical terms. I will pick 0.3 to be conservative in the clustering.

In [ ]:
clustering_cond = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=0.3,
    metric="cosine",
    linkage="average"
)

labels_cond = clustering_cond.fit_predict(embeddings_cond)

In [ ]:
len(labels_cond)

In [ ]:
n_cond_clusters = len(np.unique(labels_cond))
print(n_cond_clusters)

From the initial 1754 distinct conditions in the dataset, I got down to 827 clusters, and I saved the labels assigned to each of the original descriptors in `label_cond`. 

This information needs to be appended as an extra column to the dataset.

As when I created the embeddings I used a list of unique conditions (1754 distinct strings for a dataset of 4013 rows), I will need to map the condition/cluster_ID to the dataset. 

In [ ]:
cond_cluster_map = pd.Series(labels_cond, index=conditions)

In [ ]:
cond_cluster_map[0:10]

As a sanity check before appending the `condition_cluster` column, I will inspect some clusters. For instance, from the output above:
* **cluster 49** seems to be linked to *depression*
* **cluster 148** seems to be linked to *allergies*
* **cluster 236** seems to be linked to *hair loss*

In [ ]:
cluster_49 = cond_cluster_map[cond_cluster_map == 49].index.tolist()
print(f"Cluster {49}: {list(cluster_49)}")

In [ ]:
cluster_148 = cond_cluster_map[cond_cluster_map == 148].index.tolist()
print(f"Cluster {148}: {list(cluster_148)}")

In [ ]:
cluster_236 = cond_cluster_map[cond_cluster_map == 236].index.tolist()
print(f"Cluster {236}: {list(cluster_236)}")

As these inspections gave reassuring results, I will proceed adding the extra column.

In [ ]:
X_reviews['condition_cluster'] = X_reviews['condition'].map(cond_cluster_map)

#### <span style="color:#F97316;">Conditions ranking</span>

I now want to identify conditions that have a high number of reviews and a suboptimal rating.

To decide which metrics to use, I am inspecting the rating distributions across the entire dataset.

In [ ]:
sns.histplot(X_reviews['rating'], bins=10, kde=True)
plt.xlabel('Rating')
plt.ylabel('Count')
plt.show()

As (across conditions), the majority of reviews seem to be positive and I am more interested in lower scores, I will favor ranking by average score rather than by median.

Using an average score will say how good/bad are ratings overall. An extremely low review will carry more weight than a mid-low rated one, but a patient scoring 6 might still be unsatisfied, and only marginally affect the average score.

As such, I will also record a separate metric, capturing the percentage of patients that gave a rating lower than 7.

In [ ]:
conditions_info = X_reviews.groupby('condition_cluster').agg(
    N_reviews=('condition_cluster', 'count'),
    avg_rating=('rating', 'mean'),
    pct_below_7=('rating', lambda x: (x < 7).mean() * 100)
).reset_index()

Despite potential unmet need, it would be difficult to trust findings for conditions with very few reviews. For this reason, before creating a global score, I will drop clusters with less than 20 reviews available.

I will anyway create a copy of the dataframe, should I decide to go back and investigate discarded conditions.

In [ ]:
conditions_info_2 = conditions_info[conditions_info['N_reviews'] >= 20]
conditions_info_2.head()

In [ ]:
conditions_info_2.shape

I still have 33 potential conditions, so I can proceed and add ranking columns. I want to rank 1, respectively:
* the condition with highest number of reviews (larger population)
* the condition with the lowest average rating (clear unmet need)
* the condition with the highest percentage of reviews < 7 (large %population not being satisfied)

In [ ]:
conditions_info_2['N_reviews_rank'] = conditions_info_2['N_reviews'].rank(ascending=False, method='min').astype(int)
conditions_info_2['avg_rating_rank'] = conditions_info_2['avg_rating'].rank(ascending=True, method='min').astype(int)
conditions_info_2['pct_below_7_rank'] = conditions_info_2['pct_below_7'].rank(ascending=False, method='min').astype(int)

conditions_info_2.head()

To get a global score, I will sum the rankings. As a starting point, all metrics will carry the same weight. Conditions with the lowest global score are those that carry more values for the current investigation. 

In the future and based on the outcome of this analysis, it might be worth testing alternative rankings, varying the weightings in this step, to assess whether alternative conditions should be prioritised instead.

In [ ]:
conditions_info_2['global_score'] = conditions_info_2['N_reviews_rank'] + conditions_info_2['avg_rating_rank'] + conditions_info_2['pct_below_7_rank']
conditions_info_2.head()

I will now sort by lowest global score, and inspect the top 10 clusters.

In [ ]:
conditions_info_2 = conditions_info_2.sort_values('global_score')
conditions_info_2.head(10)

In [ ]:
top10_clusters = conditions_info_2.head(10).index.tolist()
top10_clusters

In [ ]:
for cluster_id in top10_clusters:
    members = cond_cluster_map[cond_cluster_map == cluster_id].index.tolist()
    print(f"Cluster {cluster_id}: {list(members)}")

Clustering seems to have worked well, as the conditions recorded under each of the top clusters are consistent:

* Cluster 202: *contraception*
* Cluster 90: *sinusitis*
* Cluster 131: *stop smoking aid*
* Cluster 49: *depression*
* Cluster 187: *UTI*
* Cluster 66: *hypercholesterolemia*
* Cluster 27: *acne*
* Cluster 16: *hypertension*
* Cluster 14: *rosacea*
* Cluster 13: *fibromyalgia*

The next step will be to create a subset of the reviews dataframe only containing the top 10 conditions, and rename the clusters according to the legend above, for readability.

In [ ]:
X_top10_cond = X_reviews[X_reviews['condition_cluster'].isin(top10_clusters)]

In [ ]:
cluster_labels = {
    202: "Contraception",
    90: "Sinusitis",
    131: "Stop Smoking Aid",
    49: "Depression",
    187: "UTI",
    66: "Hypercholesterolemia",
    27: "Acne",
    16: "Hypertension",
    14: "Rosacea",
    13: "Fibromyalgia"
}

X_top10_cond['cluster_name'] = X_top10_cond['condition_cluster'].map(cluster_labels)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharey=True)
axes = axes.flatten()

for ax, cluster_id in zip(axes, top10_clusters):
    subset = X_top10_cond[X_top10_cond['condition_cluster'] == cluster_id]
    cluster_name = subset['cluster_name'].iloc[0]
    sns.countplot(x=subset['rating'], ax=ax)
    ax.set_title(f'{cluster_name}')
    ax.set_xlabel('')

For the purposes of this analysis, looking at numbers of reviews available and their ratings distribution, **depression** seems to offer a good dataset to extrapolate insights. 

The next step will be to extract reviews linked to this condition, and proceed with further analysis.

Other conditions seemed interesting given the high percentage of reviews with scores = 1, however the small available datasets would make findings less robust, and prevent to safely anonimise findings from individual reviews.

In [ ]:
depression_df = X_top10_cond[X_top10_cond['condition_cluster']==49]

In [ ]:
depression_df.info()

### <span style="color:orange;">Condition Investigation</span> 

In this section, the dataset for the selected condition will be interrogated, in preparation for reviews analysis.

#### <span style="color:#F97316;">Subsets based on ratings</span>

The depression dataset has 325 entries. It can be useful to split the dataset based on review rating groups. 

I will create the following subsets:
* *depression_low* -> reviews with ratings < 5
* *depression_mid* -> reviews with ratings between 5 and 7 included
* *depression_high* -> reviews with ratings > 7

In [ ]:
depression_low = depression_df[depression_df['rating']<5]
len(depression_low)

In [ ]:
depression_mid = depression_df[(depression_df['rating']>4)&(depression_df['rating']<8)]
len(depression_mid)

In [ ]:
depression_high = depression_df[depression_df['rating']>7]
len(depression_high)

I will now visualize Effectiveness and Side Effect rankings across the three subsets.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 12))
axes = axes.flatten()

# Define sources for each subplot (df, column, graph title)
sources = [
    (depression_low, 'effectiveness', 'Rating < 5'),
    (depression_low, 'sideEffects', 'Rating < 5'),
    (depression_mid, 'effectiveness', 'Rating 5-7'),
    (depression_mid, 'sideEffects', 'Rating 5-7'),
    (depression_high, 'effectiveness', 'Rating > 7'),
    (depression_high, 'sideEffects', 'Rating > 7'),
]

# Collect every unique category across all sources, to standardize the color palette
all_categories = sorted(set().union(*[source_df[col].unique() for source_df, col, name in sources]))
palette = sns.color_palette('BuPu', n_colors=len(all_categories))
category_to_color = dict(zip(all_categories, palette))

# Plot the piecharts
for ax, (source_df, col, name) in zip(axes, sources):
    counts = source_df[col].value_counts()
    colors = [category_to_color[cat] for cat in counts.index]   # look up by category name to keep color assignment
    counts.plot.pie(autopct='%1.1f%%', ax=ax, colors=colors)
    ax.set_ylabel('')
    ax.set_title(f'{name} — {col}')
plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('data/depression_effectiveness_sideEffects.png', dpi=300, bbox_inches='tight')

Results seem reasonable, and low ratings seem to come from a mixture of low efficacy and side effects impact (giving two potential strategies for development of novel treatments):

* **Ratings < 5** -> mostly ineffective to moderate efficacy (75.8%); 66.2% of side effects are recorded as severe or extremely severe;
* **Ratings 5-7** -> mostly moderate to considerable efficacy (86.9%); 72.7% of side effects are recorded as mild to moderate;
* **Ratings >7** -> 97.5% show considerable to high efficacy; 86.6% records no to mild side effects.


#### <span style="color:#F97316;">Subsets based on efficacy and side effects</span>

As shown in the previous figure, despite the different distributions of efficacy and recorded side effects based on review ratings, there are overlaps in categories.

For this reason, I decided to create other subsets of `depression_df`, based on classifications in these features. 

I will create the following subsets:
* *depression_low_efficacy* -> reviews with efficacy being *Ineffective* or *Marginally Effective* 
* *depression_side_effects* -> reviews with side effects being moderate to extremely severe

Please note that a review can fall in both categories, so these datasets won't be independent.

In [ ]:
depression_low_efficacy = depression_df[depression_df['effectiveness'].isin(['Ineffective', 'Marginally Effective'])]
len(depression_low_efficacy)

In [ ]:
depression_side_effects = depression_df[depression_df['sideEffects'].isin(['Moderate Side Effects', 'Severe Side Effects', 'Extremely Severe Side Effects'])]
len(depression_side_effects)

It would be interesting at this point to investigate whether any particular drug comes up frequently in either of these categories. 

This would address two of the goals set out for the present investigation: gathering information on the drugs for which reviews are available in the dataset, and exploring potential correlations between specific treatments and low efficacy or presence of side effects.

In [ ]:
drugs_list = depression_df['urlDrugName'].unique().tolist()
drugs_list = sorted(drugs_list)
len(drugs_list)

Here will be listed drugs for which available reviews are present in the whole `depression_df` dataset:

In [ ]:
print(sorted(drugs_list))

I want to now visualize the drugs with most reviews across the whole dataset. Additionally, for each drug I want to capture the number of reviews that reported *low_efficacy* or *side_effects*.

In [ ]:
drugs_counts = pd.DataFrame({'Drug': drugs_list})
drugs_counts['Overall'] = drugs_counts['Drug'].map(depression_df['urlDrugName'].value_counts()).fillna(0).astype(int)
drugs_counts['Low Efficacy'] = drugs_counts['Drug'].map(depression_low_efficacy['urlDrugName'].value_counts()).fillna(0).astype(int)
drugs_counts['Side Effects'] = drugs_counts['Drug'].map(depression_side_effects['urlDrugName'].value_counts()).fillna(0).astype(int)
drugs_counts = drugs_counts.sort_values('Overall', ascending=False)
drugs_counts[drugs_counts['Overall'] > 20].head()

I will add columns to represent percentages of negative reviews for each treatment.

In [ ]:
drugs_counts['Pct Low Efficacy'] = (drugs_counts['Low Efficacy']*100/drugs_counts['Overall']).round(2)
drugs_counts['Pct Side Effects'] = (drugs_counts['Side Effects']*100/drugs_counts['Overall']).round(2)
drugs_counts[drugs_counts['Overall'] > 20].head()

I will now only consider drugs for which more than 10 reviews are present in the dataset, and sort by percentages of reviews noting side effects, then by percentages of reviews reporting low efficacy.

In [ ]:
top_drugs = drugs_counts[drugs_counts['Overall']>10]
top_drugs.sort_values(
    by=["Pct Side Effects", "Pct Low Efficacy"],
    ascending=[False, False]
)

There seem to be treatments associated with higher percentages of moderate-to-severe side effects. 

However, these data can't be used for a proper correlation analysis or to extract additional information beyond the displayed analytics, as the dataset is very small, it doesn't capture patients history, and medical context is missing on how the treatments compare (e.g. if they are administered as a firt-line of help, or based on specific criteria).

It could be anyway interesting to see if there are drugs where no reports of low efficacy or side effects are available.

In [ ]:
top_drugs_pct = top_drugs.drop(columns=['Overall', 'Low Efficacy', 'Side Effects'])

fig, ax = plt.subplots()
sns.scatterplot(data=top_drugs_pct, x='Pct Low Efficacy', y='Pct Side Effects', legend=False, ax=ax)

for _, row in top_drugs_pct.iterrows():
    ax.text(row['Pct Low Efficacy'], row['Pct Side Effects'], row['Drug'], fontsize=10, ha='left', va='bottom')

ax.set_ylim(0, 80)
ax.set_yticks(range(0, 81, 20))
ax.grid(axis='y')
ax.set_xlim(0, 40)
ax.set_xticks(range(0, 41, 10))
ax.grid(axis='x')
plt.show()

In [ ]:
fig.savefig('data/depr_drugs_effectiveness_SideEffects.png', dpi=300, bbox_inches='tight')

Amongst the drugs with at least 10 reviews, there seem to be treatment options for which only small percentages of patients report low efficacy or presence of moderate-to-severe side effects.

As mentioned above, this is meant to be purely an exercise to extract information from the available data, and cannot be treated as a real insight, given the size of the dataset and the lack of additional information.

### <span style="color:orange;">Reviews Analysis</span> 

In this section, (groups of) reviews from the selected condition will be analysed to find trends.

Three approaches (of increasing sophistication) will be used and results compared:
1. Content from subsets of reviews will be tokenized and token distributions visualised in WordCloud (initial topic overview)
2. BERTopic will be employed on the full set of reviews to find recurring themes amongst Side Effects
3. A local LLM will be employed to normalise the full set of reviews, then BERTopic adopted to cluster and define topics. A final LLM-call will be used to summarise findings

#### <span style="color:#F97316;">Low efficacy reviews: WordCloud analysis</span>

In [ ]:
depression_low_efficacy.columns

This dataset is a collection of reviews that recorded low efficacy, so it is likely that the `benefitsReview` feature will not hold much information. I will instead start interrogating the `sideEffectsReview` column to gather insights on recurring tokens.

I will use `token_list` and `cloud_plot` functions from `src`

In [ ]:
dle_t_list = token_list(depression_low_efficacy,'sideEffectsReview')
cloud_plot(dle_t_list)

There is a lot of noise in this visualization. It is clear that some tokens (such as *day* or *felt*) don't add value at this stage, so I will make a couple of manual attents at cleaning the list, removing noise from the previous visualization.

In [ ]:
dle_t_list_2 = [t for t in dle_t_list if t not in ['side', 'effect', 'day', 'week', 'felt', 'extreme', 'severe', 'time', 'drug', 'taking', 'medication']]
cloud_plot(dle_t_list_2)

Some themes start to appear, however I will do a final pass to the list before saving the results.

In [ ]:
dle_t_list_3 = [t for t in dle_t_list_2 if t not in ['became', 'stop', 'even', 'became', 'constant', 'treatment', 'made', 'thing', 'doctor', 'felt', 'really', 'take', 'still']]
fig = cloud_plot(dle_t_list_3)

In [ ]:
fig.savefig('data/depression_low_efficacy_side_Effects_wc.png', dpi=300, bbox_inches='tight')

I will now use the `tokens_freq_df` function from `src` to visualise the high-frequency tokens.

In [ ]:
dle_t_df = tokens_freq_df(dle_t_list_3)
dle_t_df.head(20)

Now it is possible to see some words that clearly enough point to undesired side effects, such as:
* anxiety
* headaches
* nausea
* dizziness
* weight gain
* affected libido

I will now repeat the analysis on the `commentsReview` feature, to see whether it might carry additional signal.

In [ ]:
dle_t_list_comments = token_list(depression_low_efficacy,'commentsReview')
cloud_plot(dle_t_list_comments)

Even in smaller prints, it is difficult to spot any additional information, better to directly move on to BERTopic and other investigations. 

#### <span style="color:#F97316;">Side Effect reviews: WordCloud analysis</span>

In this dataset, I expect to find similar / more sever side effects. However, I am also interested in analysing possible benefits that could be listed: a patient might find treatment efficient and yet might have to stop it due to sever side effects.

In [ ]:
depression_side_effects.columns

In [ ]:
dse_t_list_benefit = token_list(depression_side_effects,'benefitsReview')
cloud_plot(dse_t_list_benefit)

One of the goals set out at the start of the project was to get an overview of benefits for reviews where considerable side effects were reported. 

Few tokens here suggest that patients might have noticed benefits from treatment (e.g. *better* or *helped*) despite the side effects, however this tokens set is better suited for a more sophisticated analysis that is context-aware. 

This approach will be deprioritised at the moment, and I will move on to side effects, where I feel it might be easier to identify common themes and compare findings across methodologies.

In [ ]:
dse_t_list = token_list(depression_side_effects,'sideEffectsReview')
cloud_plot(dse_t_list)

Again, I will remove some noise and repeat the visualisation.

In [ ]:
dse_t_list_2 = [t for t in dse_t_list if t not in ['medication', 'side', 'effect', 'severe', 'taking', 'day', 'time', 'week', 'extreme', 'treatment', 'night', 'month', 'drug']]
cloud_plot(dse_t_list_2)

In [ ]:
dse_t_list_3 = [t for t in dse_t_list_2 if t not in ['feeling', 'felt', 'feel']]
fig = cloud_plot(dse_t_list_3)

In [ ]:
fig.savefig('data/depression_se_side_effects_wc.png', dpi=300, bbox_inches='tight')

In [ ]:
dse_t_df = tokens_freq_df(dse_t_list_3)
dse_t_df.head(20)

Similar tokens to the "low_efficacy" analysis are popping up. This is reasonable also because there will be an overlap in reviews (people not finding the drug efficaceous and experiencing severe side effects).
A new possible theme seems to be emerging, but it is difficult to interprete from single tokens (feel/feeling/felt).

Lastly, I will quickly interrogate the comments feature.

In [ ]:
dse_t_list_comment = token_list(depression_side_effects,'commentsReview')
cloud_plot(dse_t_list_comment)

Also in this case, it is difficult to extract meaning from single tokens from the Comment section in the reviews.

#### <span style="color:#F97316;">BERTopic approach</span>

Clustering could become unstable with small datasets such as reviews from `depression_side_effects` or `depression_low_efficacy` dataframes. For this reason, the entire `depression_df` will be used to extract topics.

In [ ]:
len(depression_df)

In `src`, I created a function to build a generic BERTopic model tailored for words embeddings for a dataset with only ~300 documents (`build_topic_model`). 

For the UMAP model, I set `min_dist` to 0.02 to create tighter clusters, and `metric` to *cosine*, as sentence transformers push semantically similar words/sentences to have high cosine similarities (also, embeddings live in high-dimensional spaces, where raw Euclidean distances don't perform well).

For the HDBSCAN model, the `min_cluster_size` was lowered to 5 (from a default of 15), given the small number of reviews. However, `min_samples` was lowered to 2 to consider as clusters data that is less densely packed (again, to favour clustering).

For the CountVectorizer model, English stopwords were removed, sentences turned lowercase and punctuation and words with less than 3 characters stripped. Finally, `ngram_range` was tailored to accept bigrams such as "side effect" or "weight loss".




#### <span style="color:#F97316;">BERTopic Side Effects</span>

First, I will create embeddings for the feature *sideEffectsReview* in the `depression_df` dataset.

In [ ]:
depression_se_list = depression_df['sideEffectsReview'].tolist()

In [ ]:
depression_se_embeddings = embedder.encode(depression_se_list, show_progress_bar=True, batch_size=32)

In [ ]:
np.save('data/depression_se_embeddings.npy', depression_se_embeddings)

In [ ]:
# Reload embeddings
depression_se_embeddings = np.load("data/depression_se_embeddings.npy")

Now, I will `fit_transform` the model using the list of reviews and their embeddings.

In [ ]:
depression_se_topic_model = build_topic_model(embedder,SEED)
topics_d_se, probs_d_se = depression_se_topic_model.fit_transform(depression_se_list, embeddings=depression_se_embeddings)

I will try to evaluate the results using the hierarchical visualization from BERTopic.

In [ ]:
depression_se_topic_model.visualize_hierarchy(width=1000, height=800)

It is possible to guess themes in some clusters (e.g. clusters 5-10-17 seem to come from positive reviews with no/mild side effects), and to pick some of the keywords already outlined in the WordCloud representations (such as anxiety, weight gain or decrease in libido).

However, to get a better understanding of the clusters, I will visualize their full labels and 10 representations.

In [ ]:
depression_se_topic_info = depression_se_topic_model.get_topic_info().drop(columns=['Representative_Docs'])

with pd.option_context('display.max_colwidth', None):
    display(depression_se_topic_info)    

Results are clear enough to give insights on recurring side effects, and to consolidate what found from the WordClouds.

Using bigrams also gives a better picture: *decrease sexual/sexual desire/decreased libido*, *weight loss/weight gain* and *elevated heart* give more robust clues than *sexual*, *weight* or *heart*.

A limitation, that is likely making clustering less reliable, is that each review will only be assigned to one cluster (although it might list multiple side effects).


#### <span style="color:#F97316;">BERTopic Benefits</span>

Now I will repeat the analysis looking at the *benefitsReview* feature, starting with the embeddings creation.

In [ ]:
depression_ben_list = depression_df['benefitsReview'].tolist()

In [ ]:
depression_ben_embeddings = embedder.encode(depression_ben_list, show_progress_bar=True, batch_size=32)

In [ ]:
np.save('data/depression_ben_embeddings.npy', depression_ben_embeddings)

In [ ]:
# Reload embeddings
depression_ben_embeddings = np.load("data/depression_ben_embeddings.npy")

Now, I will `fit_transform` the model using the list of reviews and their embeddings.

In [ ]:
depression_ben_topic_model = build_topic_model(embedder,SEED)
topics_d_ben, probs_d_ben = depression_ben_topic_model.fit_transform(depression_ben_list, embeddings=depression_ben_embeddings)

As before, I will try to evaluate the results using the hierarchical visualization from BERTopic.

In [ ]:
depression_ben_topic_model.visualize_hierarchy(width=1000, height=800)

As benefits will have less identifiable keywords than side effects, interpretation here is more difficult. The top level split seems to capture positive reviews on the top, and negative ones at the bottom.

I'll try to visualize the Representations to gather more insights.

In [ ]:
depression_ben_topic_info = depression_ben_topic_model.get_topic_info().drop(columns=['Representative_Docs'])

with pd.option_context('display.max_colwidth', None):
    display(depression_ben_topic_info)


Bigrams are particularly useful in this case, where individual words don't give much information: *lessened anxiety*, *improvement mood* and *noticed change* start building fuller pictures.

Specific drug names seem to appear in specific clusters (also noticed in the previous analysis). Possibly, similar information could be used on a larger dataset with additional metadata on patient clinical history, to rapidly identify benefits or side effects primarily associated with a specific treatment.

#### <span style="color:#F97316;">LLM/BERTopic/LLM Approach</span>

The idea behind this approach is that, if reviews are somewhat standardised using an LLM before creating embeddings, results might be cleaner, easier to cluster and to interprete/summarize.

As the dataset I'm using can't be distributed to third parties, I will use `AutoModelForCausalLM.from_pretrained` and `AutoTokenizer.from_pretrained` to run locally.

I will use `Phi-3.5 Mini Instruct` model for text generation, being a lightweight model, and the associated tokenizer.

In [ ]:
model_llm = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3.5-mini-instruct",
    device_map="auto",
    dtype="auto"
)

In [ ]:
tokenizer_llm = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct")

In [ ]:
pipe = pipeline(
    "text-generation",
    model=model_llm,
    tokenizer=tokenizer_llm,
)

In [ ]:
generation_args = {
    "max_new_tokens": 30,     # max 30 tokens
    "return_full_text": False,  # returns only generated text, not the prompt
    "do_sample": False,         # deterministic output
}

In [ ]:
d_se_list = depression_df['sideEffectsReview'].tolist()

In [ ]:
topics_llm = []

iter = 0

for review in d_se_list:
  
  if len(str(review)) < 1000:

    prompt_1 = [
    {"role": "system", "content": "You work as a data analyst in drug discovery and you want to find common side effects from user reviews. Return the main 3 topics as a JSON list of strings, for example: ['Topic 1', 'Topic 2', 'Topic 3']."},
    {"role": "user", "content": "In the following review, pick out the main 3 topics: The drug is giving me headaches, I sleep badly and I put on a few pounds since I started treatment."},
    {"role": "assistant", "content": "['Headaches', 'Sleep disruption', 'Weight gain']"},
    {"role": "user", "content": f"In the following customer review, pick out the main 3 topics: {review}"}
    ]

    output = pipe(prompt_1, **generation_args)
    topic_list_string = output[0]['generated_text'].replace("'", '"')

    try:
        topic_list = json.loads(topic_list_string)

    except json.JSONDecodeError: # relevant if the model didn't follow the exact output format

        cleaned_string = re.sub(r'^\d+\.\s*', '', topic_list_string, flags=re.MULTILINE)

        temp_topics = [item.strip() for item in cleaned_string.split('\n') if item.strip()]

        if len(temp_topics) == 1: # If only one line, try splitting by comma
            temp_topics = [item.strip() for item in temp_topics[0].split(',') if item.strip()]

        # Take at most 3 topics, or default to the original string as one topic if nothing is extracted
        topic_list = temp_topics[:3] if temp_topics else [topic_list_string.strip()]


    topics_llm.append(topic_list)

    iter += 1

    print(iter)

In [ ]:
# Save
with open('data/topics_llm.json', 'w') as f:
    json.dump(topics_llm, f, indent=2)

In [ ]:
# Reload
with open('data/topics_llm.json', 'r') as f:
    topics_llm = json.load(f)

I noticed that 5 reviews were not processed (325 reviews available, yet it only completed the task 320 times). This is likely due to the fact that I selected to ignore reviews longer than 1000 characters (as extraction quality on long passages might be poor for lightweight models). 

Given the small number of reviews that were missed out, and due to the fact that it took nearly 3h to complete this step as is, I will proceed with the analysis regardless.

As at this stage we are still at "single-review" level, I will not openly display the results. However, I carried out topics inspection to assess whether the formatting ['side effect 1','side effect 2','side effect 3'] was followed.

From the inspection, I noticed adherence to the prompt instructions in several cases. However, a common pattern that I found was returning something in the following formats: 

* ['["side effect 1","side effect 2","side effect 3"]','Here's a breakdown of the topics:']
* ['["side effect 1"','"side effect 2"','"side effect 3"']

To fix these cases without corrupting the other entries, I followed the following approach for entries where double-quotes ("") were present :

1. `''.join(entry)` was used to first join all strings listed in the entry, giving the following ->
    * '["side effect 1","side effect 2","side effect 3"]Here's a breakdown of the topics:'
    * '["side effect 1""side effect 2""side effect 3"'

2. `re.findall(pattern, string)` with pattern `r'"([^"]+)"'` was used to read the obtained string and retrieved all that is between double-quotes ("") ->
    * ['side effect 1','side effect 2','side effect 3']
    * ['side effect 1','side effect 2','side effect 3']

The corresponding function `fix_topics_list` sits in `src`.

In [ ]:
topics_llm_fixed = [fix_topic_list(entry) for entry in topics_llm]

I will now flatten the list, use Counter to preserve information of side effects frequency, then extract the unique topics for use in BERTopic.

In [ ]:
flattened_topics = []
for sublist in topics_llm_fixed:        # each inner list
    for topic in sublist:          # each item within that inner list
        flattened_topics.append(topic)

In [ ]:
topic_counts = Counter(flattened_topics) 

In [ ]:
unique_topics = list(topic_counts.keys())

Upon inspection, most side effects look ok, with the exception of 3 phrases badly parsed. I will now drop them.

In [ ]:
junk_topics = {
    'Since the provided text does not contain any specific topics',
    'but rather a general statement',
    'it"s not possible to extract distinct topics. However',
}

clean_topic_counts = {topic: count for topic, count in topic_counts.items() if topic not in junk_topics}
clean_unique_topics = [t for t in unique_topics if t not in junk_topics]

Saving the aggregate list and dict-like counter.

In [ ]:
# unique_topics — plain list of strings
with open('data/clean_unique_topics.json', 'w') as f:
    json.dump(clean_unique_topics, f, indent=2)

# topic_counts — a Counter (dict-like), convert to a plain dict first
with open('data/clean_topic_counts.json', 'w') as f:
    json.dump(dict(clean_topic_counts), f, indent=2)

In [ ]:
#with open('data/clean_unique_topics.json', 'r') as f: clean_unique_topics = json.load(f)
#with open('data/clean_topic_counts.json', 'r') as f: clean_topic_counts = json.load(f) 

I will now proceed to generate the embeddings.

In [ ]:
llm_topic_embeddings = embedder.encode(clean_unique_topics, show_progress_bar=True, batch_size=32)

In [ ]:
np.save('data/llm_topic_embeddings.npy', llm_topic_embeddings)

In [ ]:
# Reload
llm_topic_embeddings = np.load('data/llm_topic_embeddings.npy')

BERTopic will be used to assign labels to the clusters.

In [ ]:
llm_topic_model = build_topic_model(embedder,SEED)
topics_bert_llm, probs_bert_llm = llm_topic_model.fit_transform(clean_unique_topics, embeddings=llm_topic_embeddings)

As usual, I will start investigating the clusters displaying a topic dendrogram.

In [ ]:
llm_topic_model.visualize_hierarchy(width=1000, height=800)

This approach gives clearly more granularity compared to the previous approach, where BERTopic was performed directly on the raw reviews. 

In the previous approach, the input to the model was a list of reviews, each producing an embedding. Here, from each reviews, (multiple) side effect keywords are extracted first, and BERTopic receives a cleaner list of keywords to generate embeddings. Multiple reviews might cover different topics: with the first approach, clusters would match reviews with the same combinations of side effects, whilst here side effect in each review are treated independently. 

Looking at keywords for each cluster, labels seem to contain less noise. I will confirm this visualising the representations of the clusters.

In [ ]:
llm_topic_model_info = llm_topic_model.get_topic_info().drop(columns=['Representative_Docs'])

with pd.option_context('display.max_colwidth', None):
    display(llm_topic_model_info)


Some clusters (e.g. topics 7 and 11) show that for multiple reviews the LLM didn't follow the topics formatting, and parts of sentences got categorized as keywords (as notices earlier in `junk_topics`). 

However, results show clusters of words that seem to have meaning. 

Given the added granularity (45 clusters vs the 24 previously obtained with the *direct-BERTopic* approach on the same reviews set), it becomes increasingly useful to add another LLM prompt to better define the clusters labels, rather than reviewing each cluster manually.

In [ ]:
iter = 0
llm_labels = []

for topic in llm_topic_model_info['Representation']:


    prompt_2 = [
    {"role": "system", "content": "You work as a clinician and you want to understand what medical symptom is described by lists of keywords. If the keywords don't clearly link to a precise symptom, return a string that says 'Unclear'. Otherwise return a string that gives a clinical label for the symptom."},
    {"role": "user", "content": "Pick a label for the following list of keywords: [sleepiness, fatigue sleepiness, morning, fatigue, grogginess, work, tiredness, excessive, persistent, drowsiness]"},
    {"role": "assistant", "content": "'Excessive Daytime Sleepiness (EDS)'"},
    {"role": "user", "content": "Pick a label for the following list of keywords: [effects effects, effects, mention, customer review, review, customer, effects long, major, provided customer, negative]"},
    {"role": "assistant", "content": "'Unclear'"},
    {"role": "user", "content": f"Pick a label for the following list of keywords: {topic}"}
    ]

    output = pipe(prompt_2, **generation_args)
    llm_labels.append(output[0]['generated_text'])

    iter += 1
    print(iter)

In [ ]:
np.save('data/llm_labels.npy', llm_labels)

In [ ]:
llm_labels = np.load('data/llm_labels.npy')

In [ ]:
llm_topic_model_info['LLM Label'] = llm_labels

In [ ]:
with pd.option_context('display.max_colwidth', None):
    display(llm_topic_model_info)

In this case, the LLM consistently appended additional commentary after the cluster label. I will clean the label column stripping everything that follows the *\n\n* pattern

In [ ]:
llm_topic_model_info['LLM Label'] = (
    llm_topic_model_info['LLM Label']
    .str.split('\n\n').str[0]
    .str.strip()
    .str.strip("'")
)

Also, in several cases, errors in formatting resulted in labels starting with "The ". In these cases, due to the string length limit I imposed, it is difficult to extrapolate the meaning. I will rename those to "Formatting Error"

In [ ]:
llm_topic_model_info.loc[
    llm_topic_model_info['LLM Label'].str.startswith(('The ', 'Given')), 'LLM Label'
] = 'Formatting Error'

In [ ]:
with pd.option_context('display.max_colwidth', None):
    display(llm_topic_model_info)

Due to the high number of formatting errors, and some labels that don't seem to be completely grounded by the representations, I would consider this approach as failed.

My last attempt at using LLM prompting will be to generate a list of top 10 clinical symptoms that can be extracted from the topic name column.

I will start preparing the input for the prompt.


In [ ]:
cluster_list_text = "\n".join(
    f"{i+1}. {row['Name']} (mentioned in {row['Count']} reviews)"
    for i, row in llm_topic_model_info.iterrows()
    if row['Topic'] != -1   # exclude the outlier/noise cluster
)
cluster_list_text

In [ ]:
prompt_3_system_content = (
    "You are a clinician reviewing a list of symptom-related topic clusters extracted "
    "from patient drug reviews. Each numbered line represents one distinct symptom "
    "cluster — treat the words within a single line as describing one cohesive symptom "
    "concept, not as independent items. Identify the 10 most clinically significant and "
    "prevalent symptoms represented in this list, using the review counts as a guide to "
    "prevalence. Return ONLY a JSON list of exactly 10 strings, with no explanation, "
    "commentary, or additional text."
)

prompt_3 = [
    {"role": "system", "content": prompt_3_system_content},
    {"role": "user", "content": cluster_list_text}
]

generation_args_summary = {**generation_args, "max_new_tokens": 150}
output = pipe(prompt_3, **generation_args_summary)
raw_text = output[0]['generated_text']
print(raw_text)

In [ ]:
np.save('data/raw_text.npy', raw_text)

This approach successfully returned 10 top side effects, based on the clustering outputs of BERTopic applied to the initial LLM-derived topics list.

The ten highlighed side effects are:
* *decreased sexual performance*
* *withdrawal symptoms*
* *fatigue*
* *stomach issues*
* *memory loss*
* *weight change*
* *irritability*
* *treatment efficacy* (this is likely coming from noisy clusters)
* *appetite loss*
* *sleep disturbances*

Most themes seem in line with previous visualisations from NTLK and direct-BERTopic approaches, and overall seem to better find and describe clusters topics.

The added clarity comes at a computational cost, as the LLM-prompt that rearranged phrasing of individual reviews took by itself 3h on a small dataset (approx 300 reviews).

For larger datasets this approach would need to be revisited. Another limitation came from the fact that I used a local lightweight model (to prevent redistribution of the data to third parties). Where possible, using larger model would improve the outcome and speed up the analysis.

### <span style="color:orange;">Discussion</span>

The goal of this project was to use the [Drug Reviews](https://doi.org/10.24432/C55G6J) dataset to build a pipeline that could identify medical conditions for which, despite treatments being available, there is remaining unmet clinical need.

The dataset was initially prepared by removing reviews lacking commentary or condition information, and by clustering different wordings of the same conditions. This was achieved using *all-mpnet-base-v2* sentence transformer and `AgglomerativeClustering` algorithm.

Following, conditions where at least 20 reviews were available were ranked based on a global score. This score gave equal weight to three metrics: number of available reviews, average reviews rating and percentage of reviews with a rating lower than 7. Review ratings distribution of the 10 conditions with the lowest global scores were investigated.

**Depression** was selected for further analysis, given the relatively high number of reviews (325), and for the associated rating distribution, that showed a substantial number of very poor ratings. 

An initial investigation was conducted interrogating `effectiveness` and `sideEffects` features to better understand the area of unmet need. Reviews were divided in three groups based on their ratings, and scores distributions for these two features were compared across the subgroups. Low ratings seemed to come from a mixture of low efficacy and side effects impact, giving two potential strategies for development of novel treatments.

The scope of the analysis was to address the following points for the selected condition:
1. information on competitor drugs where available in the dataset (e.g. list of treatments used by reviewers)
2. possible correlations between a side effect and a specific treatment
3. interrogation on benefits reported from treatment (information on needs currently met)
4. summary of (potentially recurrent) side effects where applicable


**Competitor Drugs Analysis (addressing 1 and 2)**

It has to be noted that this dataset didn't include any information on patient history or on the specific treatment, so direct comparison of reviews for different drugs should not be considered as a meaningful finding without further investigations. However, the number of different treatments mentioned across the 325 reviews (33) shows by itself a very crowded competitive landscape, something to be considered when evaluating new developments.

For each treatment, percentage of reviews scoring no/marginal efficacy and percentage of reviews scoring moderate to extremely severe side effects were calculated. Amongst the drugs with at least 10 reviews, *Wellbutrin-XL* appeared to be a treatment option that was had no associated negative reviews in terms of efficacy, and the lowest percentage of patients reporting moderate-to-severe side effects in the dataset (13.64%). Should this result be confermed in a larger, up-to-date and more complete dataset, this could become a good benchmark option for novel drug discovery campaigns.

Five treatments appeared to be associated with an higher incidence of side effects, with 50% or more patients reporting moderate-to-extreme adverse events.

**Benefits and Side Effect Reviews Analysis (addressing 3 and 4)**

Different approaches were used to extract insights from the reviews.

Initially, Benefits, Comments and Side Effect columns were investigated using WordCloud visualisations on both datasets (reviews that scored low in efficacy and reviews with considerable side effects), iteratively to manually remove irrelevant words. It has to be noted that the two datasets are not independent, as some reviews can belong to both subsets.

For both datasets, not much could be gathered from Benefits and Comments features, where single words might not be sufficient to capture the meaning of a sentence. Side effects include tokens that are clear keywords, and can be sufficient to get more information.

Tokens that were frequently highlighted in the side effect columns of both sets were:
* nausea
* affected libido (sexual, libido)
* sleep disturbances (sleep)
* weight changes (weight)

Moving on, BERTopic was used on the full `depression_df` dataset, as clustering could become unstable on smaller datasets. Tokens and bigrams were clustered based on cosine similarity.

Analysis on the Side Effect feature revelaed recurring side effects similar to those found from the WordClouds. Using bigrams also gave a better picture: *decrease sexual/sexual desire/decreased libido*, *weight loss/weight gain* and *elevated heart* give more robust clues than *sexual*, *weight* or *heart*.

Representations from the analysis included:
* affected libido (*0_effects_possible_loss_libido*, *4_sexual_decrease sexual_sexual desire_decrease*)
* digestion issues (*3_nausea_stomach_diarrhea_food*)
* sleep disturbances and tiredness (*6_drowsiness_lack_irritability_anxiety*, *7_insomnia_energy_did_weeks*, *11_patient_morning_tired_night*)
* hyperarousal (*23_aggression_sensitive_type_tempered*, *6_drowsiness_lack_irritability_anxiety*, *18_heart_condition_constipation_elevated heart*)
* tinnitus (*9_tinnitus_severe_did experience_experience*)
* withdrawal symptoms (*14_dosage_withdrawal_physical_loss libido*)
* dry mouth (*16_dry_mouth_dry mouth_lot*)
* weight changes (*19_gain_weight gain_tired_weight*, *21_cravings_carbohydrates_headaches_occasional*)

As visible by the representations containing multiple side effects for the same cluster, a limitation of this approach is that each review is only assigned to one cluster, despite possibly listing multiple side effects. This reduces readibility of the cluster representation, and causes loss in information.

The analysis on the Benefits feature was not as straightforward, as most representations didn't have clearly identifiable themes. Bigrams however were particularly useful in this case, where individual words didn't give much information: *lessened anxiety*, *improvement mood* and *noticed change* started to build a fuller picture.

Specific drug names seemed to appear in specific clusters. Possibly, similar information could be used on a larger dataset with additional metadata on patient clinical history, to rapidly identify benefits or side effects primarily associated with a specific treatment.

Moving on, I still tested an approach that integrated prompting of a local lightweight model, to improve outcomes.

The first prompt aimed at extracting all topics covered in each review. Topics from all reviews were appended in a unique list, and were clearly readable; however, more than 600 unique topics were extracted, so BERTopic was used next on embeddings from the list, to produce a limited number of labelled clusters.

More than 40 clusters were identified, and representations were more precised compared to the BERTopic-direct approach. Many side effects reconfirmed previous findings, with some appearing for the first time:
* withdrawal symptoms (*1_withdrawal_discontinuation_lexapro_withdrawal symptoms*)
* fatigue and sleep disturbances (*3_fatigue_drowsiness_tiredness_extreme*, *18_trouble_sleep_difficulty_increased sleep*, *21_sleep_insomnia_baseline_disturbance*, *31_sleepiness_fatigue sleepiness_morning_fatigue*, *42_napping_need_insomnia_tiredness*, *44_energy_lowered_decrease_*)
* gastrointestinal issues (*5_stomach_gastrointestinal_gastrointestinal issues_constipation*, *22_nausea_medication intake_sickness_mild nausea*)
* sexual dysfunctions (*6_night_skin_excessive_sexual effects*)
* memory problems (*8_memory_problems_psychological_loss*, *26_alteration_disturbances_insomnia_memory*)
* weight changes (*9_weight_weight loss_weight gain_gain*, *14_appetite_loss appetite_change weight_importance*, *23_body_changes_change_head*)
* hyperarousal (*10_irritability_inhibition_extreme_agitation*, *13_cardiovascular_racing_function_increase*, *20_mood_mood swings_swings_changes*, *28_electric_shakiness_racing_jitteriness*, *30_anxiety_attacks_panic attacks_nervousness*)
* headaches (*24_migraines_shakiness__*, *32_headaches_migraines_frequency_food*)
* cramps (*33_cramps_pain_leg_severe*)
* emotional dulling (*34_emotional_dulling_emotional numbness_numbness*)
* dry mouth (*35_dry_mouth_dry mouth_sleepiness lack*)

With a new prompt, cleaning/summarizing of the representations was attempted, but it produced poor results. Instead, representations and frequencies were passed to a final prompt, where the model was asked to produce a list of the top 10 side effects. This was the output:
* *decreased sexual performance*
* *withdrawal symptoms*
* *fatigue*
* *stomach issues*
* *memory loss*
* *weight change*
* *irritability*
* *treatment efficacy* (this is likely coming from noisy clusters)
* *appetite loss*
* *sleep disturbances*

Overall, using LLMs performed well to extract topics from the individual reviews, but looking at findings the gain was marginal, considering the computational cost of analysing the individual reviews. Results were cleaner, but the top side effects were already captured with the simpler methods. The limitation came from the necessity of using a local lightweight model, as the dataset could not be shared to third parties, and the LLM often disregarded formatting instructions, adding noise to the otherwise cleaner results. 

### <span style="color:orange;">Executive Summary and Conclusions</span>

Amongst the various conditions, depression was identified as potentially carrying unmet clinical need. This was due to the relatively higher percentage of poorer scores in associated reviews, although the selection was limited by the small dataset, since conditions with more reviews had to be prioritized.

Low scores were linked both to limited efficacy and to considerable size effects, potentially giving multiple approaches for future developments.

The multitude of treatments mentioned across the reviews depicts a crowded competitive landscape, and from the available reviews there seem to be treatments that offer good efficacy and low side effects. Given the limited available information, however, no direct comparisons can be made between treatments, and subsequent analysis was performed across all reviews.

Focusing on side effect reviews, similar findings were corroborated using different approaches, and the most recurring events included:
* *decreased sexual performance*
* *withdrawal symptoms*
* *fatigue and sleep disturbances*
* *gastrointestinal issues*
* *hyperarousal*
* *dry mouth*
* *headaches*

Many of these sympthoms are difficult to measure preclinically, so mechanistic drivers should be investigated to properly counter-screen in early development. Feasibility of this should be carefully considered.

Please note that this analysis is performed on a small set of drug reviews from 2018, so there is no claim for findings to be accurate or exhaustive at any extent.